# 50 · Segmentação oficial: v3 com k = 4

Decisão do grupo em 14/11/2026, depois dos testes dos notebooks 44 a 49:

- **População**: os 13.999 assinantes ativos em 30/09/2026 que treinaram nos 90 dias até o corte
  ("elegível ao k-means"). Os 754 ativos sem treino na janela entram como um quinto segmento, **por regra**
  (`segmento_regra = "inativo em 90 dias"`), fora do k-means.
- **Variáveis (v3)**, todas em z-score entre os elegíveis: treinos por semana, dias sem treino (log),
  tempo de casa (log), % dos treinos ao ar livre e engajamento (ordinal: possível churn 0 → alto 3).
- **Fora do k-means, só no perfil**: canal de aquisição (R² zero), plano (separava pouco) e km por treino
  (o 0 de quem só treina indoor formava um grupo artificial no k = 5).
- **Algoritmo**: `KMeans(n_clusters=4, n_init=10, random_state=0)`.
- **k = 4** pelas quatro lentes: silhueta 0,30 (fraca, a maior entre as versões com quatro grupos),
  estabilidade alta (Rand ≥ 0,98), a inércia ainda cai bem até o 4º grupo, e cada grupo pede uma ação
  diferente da Vitaliza.

Este notebook entrega o mínimo da entrega parcial: **a tabela** (perfil em medianas, tamanho e sinal de
saída), **o gráfico** (inércia e silhueta de k = 2 a 8 com o k escolhido marcado) e **um nome por grupo**.
Ordem de execução: 39 → 41 → 42 → 43 → 44 → … → 49 → 50.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
K = 4; SEED = 0
AR_LIVRE = ["corrida", "caminhada", "pedal"]

m = pd.read_parquet(P / "tabela_master.parquet")
print("tabela master:", m.shape)
print(m.segmento_regra.value_counts().to_string())
e = m[m.segmento_regra == "elegível ao k-means"].copy().reset_index(drop=True)
assert (e.data_referencia == CORTE).all() and e.ativa_no_corte.all()
print("\nelegíveis ao k-means:", len(e), "| contas distintas:", e.user_id.nunique())

tabela master: (63929, 119)
segmento_regra
fora do escopo (não é assinante ativo)    49176
elegível ao k-means                       13999
inativo em 90 dias                          754

elegíveis ao k-means: 13999 | contas distintas: 13999


## 1. % dos treinos ao ar livre

In [2]:
e["pct_ar_livre"] = (e[["treinos_" + t for t in AR_LIVRE]].sum(axis=1) / e.treinos_90d * 100).astype(float)
print("treinos_90d > 0 em todos os elegíveis:", bool((e.treinos_90d > 0).all()))
print(e.pct_ar_livre.describe().round(1).to_dict())
print("0% ao ar livre:", int((e.pct_ar_livre == 0).sum()), "| 100% ao ar livre:", int((e.pct_ar_livre == 100).sum()))

treinos_90d > 0 em todos os elegíveis: True
{'count': 13999.0, 'mean': 54.4, 'std': 34.6, 'min': 0.0, '25%': 16.7, '50%': 71.4, '75%': 83.3, 'max': 100.0}
0% ao ar livre: 691 | 100% ao ar livre: 1087


## 2. Km por treino

Média de `distancia_km` dos treinos ao ar livre (corrida, caminhada, pedal) na janela de 90 dias até o
corte (mesma janela da tabela master: 0 a 89 dias antes de 30/09/2026), só treinos com GPS válido e
distância maior que zero. Quem não tem nenhum treino com distância na janela fica com **0 km**.

In [3]:
a = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "distancia_km", "gps_ok"])
print("treinos no arquivo:", len(a))
a = a[a.user_id.isin(e.user_id)]
print("-> dos elegíveis:", len(a))
a = a[(CORTE - a.data).dt.days.between(0, 89)]
print("-> na janela de 90 dias:", len(a))
al = a[a.tipo.isin(AR_LIVRE)]
print("-> ao ar livre:", len(al), "| GPS falhou:", int((~al.gps_ok).sum()), "| GPS ok sem distância ou zero:", int((al.gps_ok & ~(al.distancia_km > 0)).sum()))
d = al[al.gps_ok & (al.distancia_km > 0)]
print("-> com distância válida:", len(d), "| contas:", d.user_id.nunique())
km = d.groupby("user_id").distancia_km.mean().rename("km_por_treino")
e = e.merge(km, on="user_id", how="left")
fal = set(al.user_id) - set(d.user_id)
print("\ncontas sem treino com distância (ficam com 0 km):", int(e.km_por_treino.isna().sum()))
print("  das quais têm treino ao ar livre na janela, todos sem distância válida (GPS falhou):", len(fal))
e["km_por_treino"] = e.km_por_treino.fillna(0)
e["km_por_treino_log"] = np.log1p(e.km_por_treino)
print("\nkm por treino, quem tem distância:", e.km_por_treino[e.km_por_treino > 0].describe().round(1).to_dict())
print("km por treino por tipo principal (mediana, quem tem distância):")
print(e[e.km_por_treino > 0].groupby("tipo_principal").km_por_treino.median().round(1).to_string())
print("assimetria antes e depois do log:", round(e.km_por_treino.skew(), 2), "->", round(e.km_por_treino_log.skew(), 2))

treinos no arquivo: 1252488


-> dos elegíveis: 941195
-> na janela de 90 dias: 372012
-> ao ar livre: 199229 | GPS falhou: 0 | GPS ok sem distância ou zero: 0
-> com distância válida: 199229 | contas: 13308

contas sem treino com distância (ficam com 0 km): 691
  das quais têm treino ao ar livre na janela, todos sem distância válida (GPS falhou): 0

km por treino, quem tem distância: {'count': 13308.0, 'mean': 8.7, 'std': 9.5, 'min': 1.9, '25%': 5.0, '50%': 5.8, '75%': 7.6, 'max': 162.8}
km por treino por tipo principal (mediana, quem tem distância):
tipo_principal
caminhada    5.40
corrida      7.00
forca        5.20
funcional    5.20
pedal       27.30
yoga         5.10
assimetria antes e depois do log: 6.28 -> -0.17


## 3. Engajamento, plano e canal

In [4]:
ORD = {"possível churn": 0, "baixo": 1, "médio": 2, "alto": 3}
e["engajamento_ord"] = e.nivel_engajamento.map(ORD)
assert e.engajamento_ord.notna().all()
print(e.nivel_engajamento.value_counts().to_string())
e["plano_anual"] = (e.plano == "anual").astype(int)
print("\nplano anual:", int(e.plano_anual.sum()), f"({e.plano_anual.mean():.1%})")
canal = pd.get_dummies(e.canal_aquisicao, prefix="canal").astype(int)
e = pd.concat([e, canal], axis=1)
CANAIS = list(canal.columns)
print("\ncanal:"); print(e.canal_aquisicao.value_counts().to_string())

nivel_engajamento
alto              10689
médio              2366
possível churn      579
baixo               365

plano anual: 7727 (55.2%)

canal:
canal_aquisicao
organico                4410
influenciador           2379
meta_ads                2294
indicacao               2113
google_ads              1435
parceria_corporativa    1368


## 4. Matriz da v3

In [5]:
NUM = ["treinos_por_semana", "dias_sem_treinar_log", "duracao_dias_log", "pct_ar_livre", "engajamento_ord"]
sc = StandardScaler().fit(e[NUM])
X = pd.DataFrame(sc.transform(e[NUM]), columns=[c + "_z" for c in NUM])
print("matriz:", X.shape, "| vazios:", int(X.isna().sum().sum()))
pd.DataFrame({"variavel": NUM, "media_elegiveis": sc.mean_, "desvio_elegiveis": sc.scale_}).to_csv(
    S / "segmentacao_oficial_padronizacao.csv", index=False, encoding="utf-8-sig")
print("parâmetros da padronização: saidas/segmentacao_oficial_padronizacao.csv")

matriz: (13999, 5) | vazios: 0
parâmetros da padronização: saidas/segmentacao_oficial_padronizacao.csv


## 5. O gráfico: inércia e silhueta de k = 2 a 8

Silhueta com todos os pontos em cada k.

In [6]:
rows, modelos = [], {}
for k in range(2, 9):
    mk = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    modelos[k] = mk
    rows.append({"k": k, "inercia": mk.inertia_, "silhueta": silhouette_score(X, mk.labels_), "menor_grupo": int(np.bincount(mk.labels_).min())})
curva = pd.DataFrame(rows)
curva["ganho_do_grupo_seguinte"] = -curva.inercia.diff().shift(-1)
print(curva.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(curva.k, curva.inercia, "o-", color="#222")
ax[0].set_title("Inércia (cai sempre; procurar a dobra)"); ax[0].set_xlabel("número de grupos (k)")
ax[1].axhspan(0, 0.25, color="#ddd", alpha=.5); ax[1].axhspan(0.25, 0.5, color="#fbe3e3", alpha=.6)
ax[1].text(8.4, 0.12, "sem estrutura", ha="right", fontsize=8, color="#555"); ax[1].text(8.4, 0.37, "fraca", ha="right", fontsize=8, color="#555")
ax[1].plot(curva.k, curva.silhueta, "o-", color="#222")
ax[1].set_title("Silhueta (média de todos os pontos)"); ax[1].set_xlabel("número de grupos (k)"); ax[1].set_ylim(0, 0.6)
for a_, col in zip(ax, ["inercia", "silhueta"]):
    v = curva.set_index("k").loc[K, col]
    a_.plot([K], [v], "o", color="#e5484d", ms=10, zorder=3)
    a_.annotate(f"k = {K}", (K, v), textcoords="offset points", xytext=(8, 8), color="#e5484d")
fig.suptitle("k-means da Vitaliza: 13.999 assinantes ativos que treinaram, 5 variáveis (v3)")
fig.tight_layout(); fig.savefig(S / "segmentacao_oficial_curvas.png", dpi=130)
curva.to_csv(S / "segmentacao_oficial_curva.csv", index=False, encoding="utf-8-sig")
print("salvo: saidas/segmentacao_oficial_curvas.png e saidas/segmentacao_oficial_curva.csv")

 k   inercia  silhueta  menor_grupo  ganho_do_grupo_seguinte
 2 47,979.36      0.35         3252                 9,513.13
 3 38,466.23      0.28         3069                 7,347.46
 4 31,118.77      0.30         2092                 4,384.86
 5 26,733.91      0.31          941                 3,493.69
 6 23,240.22      0.30          938                 2,250.92
 7 20,989.29      0.31          833                 1,822.78
 8 19,166.51      0.31          833                      NaN


salvo: saidas/segmentacao_oficial_curvas.png e saidas/segmentacao_oficial_curva.csv


## 6. k = 4: estabilidade e conferência com o notebook 49

In [7]:
km4 = modelos[K]; lab = km4.labels_
seeds = [adjusted_rand_score(lab, KMeans(n_clusters=K, n_init=10, random_state=s).fit_predict(X)) for s in [1, 2, 3, 4, 5]]
rng = np.random.default_rng(SEED); boot = []
for i in range(5):
    idx = rng.integers(0, len(X), len(X))
    boot.append(adjusted_rand_score(lab, KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X.iloc[idx]).predict(X)))
print("Rand com outras sementes:", [round(v, 3) for v in seeds])
print("Rand em reamostragens:", [round(v, 3) for v in boot])
n49 = pd.read_parquet(P / "kmeans_v3_k4_rotulos.parquet")
c = e[["user_id"]].assign(lab=lab).merge(n49, on="user_id")
print("conferência com o notebook 49, Rand:", round(adjusted_rand_score(c.lab, c.grupo_k4), 3))

Rand com outras sementes: [0.999, 0.999, 0.999, 0.999, 0.999]
Rand em reamostragens: [0.984, 0.986, 0.99, 0.99, 0.987]
conferência com o notebook 49, Rand: 1.0


## 7. Um nome por grupo

O nome é dado pelo perfil, não pelo número do grupo, para não depender da ordem que o k-means devolve:

- **Esfriando**: o grupo com a menor fração de engajamento alto;
- **Recém-chegados**: entre os outros, o de menor tempo de casa;
- **Ativos ao ar livre** e **Ativos em ambiente fechado**: os dois restantes, pela % de treinos ao ar livre.

Os nomes resumem a evidência e são hipóteses a testar.

In [8]:
g = e.assign(c=lab).groupby("c")
alto = g.nivel_engajamento.apply(lambda s: (s == "alto").mean())
esf = alto.idxmin()
resto = [i for i in range(K) if i != esf]
rec = g.meses_de_casa.median().loc[resto].idxmin()
dois = [i for i in resto if i != rec]
ar = g.pct_ar_livre.median().loc[dois].idxmax()
ind = [i for i in dois if i != ar][0]
NOMES = {ar: "Ativos ao ar livre", ind: "Ativos em ambiente fechado", esf: "Esfriando", rec: "Recém-chegados"}
assert len(set(NOMES.values())) == K
print(NOMES)
e["segmento"] = pd.Series(lab).map(NOMES).values

{np.int32(3): 'Ativos ao ar livre', 1: 'Ativos em ambiente fechado', np.int32(2): 'Esfriando', np.int32(0): 'Recém-chegados'}


## 8. A tabela: perfil de cada segmento

Os quatro segmentos do k-means e o quinto, por regra (ativos sem treino nos 90 dias). Medianas na unidade
original e a lente de três fatores: **ativos** (tamanho), **engajamento** (níveis) e **sinal de saída**
(renovação automática desligada no corte; não há saída observada depois de 30/09/2026).

In [9]:
ina = m[m.segmento_regra == "inativo em 90 dias"].copy()
ina["segmento"] = "Inativos em 90 dias (regra)"
ina["plano_anual"] = (ina.plano == "anual").astype(int)
ina["pct_ar_livre"] = np.nan; ina["km_por_treino"] = np.nan
cols = ["user_id", "segmento", "treinos_por_semana", "dias_sem_treinar", "meses_de_casa", "pct_ar_livre", "km_por_treino",
        "plano_anual", "nivel_engajamento", "renovacao_desligada", "canal_aquisicao", "tipo_principal"]
todos = pd.concat([e[cols], ina[cols]], ignore_index=True)
print("assinantes ativos no corte:", len(todos), "(esperado 14.753) | elegíveis:", len(e), "| inativos em 90 dias:", len(ina))
assert len(todos) == 14753 and todos.user_id.is_unique

ORDEM = ["Ativos ao ar livre", "Ativos em ambiente fechado", "Recém-chegados", "Esfriando", "Inativos em 90 dias (regra)"]
g = todos.groupby("segmento")
tab = pd.DataFrame({
    "assinantes": g.size(),
    "pct_ativos": g.size() / len(todos) * 100,
    "treinos_por_semana": g.treinos_por_semana.median(),
    "dias_sem_treinar": g.dias_sem_treinar.median(),
    "meses_de_casa": g.meses_de_casa.median(),
    "pct_ar_livre": g.pct_ar_livre.median(),
    "km_por_treino": g.km_por_treino.median(),
    "pct_anual": g.plano_anual.mean() * 100,
    "pct_engaj_alto": g.nivel_engajamento.apply(lambda s: (s == "alto").mean() * 100),
    "pct_engaj_medio": g.nivel_engajamento.apply(lambda s: (s == "médio").mean() * 100),
    "pct_engaj_baixo": g.nivel_engajamento.apply(lambda s: (s == "baixo").mean() * 100),
    "pct_possivel_churn": g.nivel_engajamento.apply(lambda s: (s == "possível churn").mean() * 100),
    "pct_renovacao_desligada": g.renovacao_desligada.apply(lambda s: s.astype(float).mean() * 100),
    "pct_influenciador": g.canal_aquisicao.apply(lambda s: (s == "influenciador").mean() * 100),
    "tipo_principal_mais_comum": g.tipo_principal.agg(lambda s: s.value_counts().index[0]),
}).loc[ORDEM]
tab.loc["Total de ativos"] = {
    "assinantes": len(todos), "pct_ativos": 100, "treinos_por_semana": todos.treinos_por_semana.median(),
    "dias_sem_treinar": todos.dias_sem_treinar.median(), "meses_de_casa": todos.meses_de_casa.median(),
    "pct_ar_livre": e.pct_ar_livre.median(), "km_por_treino": e.km_por_treino.median(), "pct_anual": todos.plano_anual.mean() * 100,
    "pct_engaj_alto": (todos.nivel_engajamento == "alto").mean() * 100, "pct_engaj_medio": (todos.nivel_engajamento == "médio").mean() * 100,
    "pct_engaj_baixo": (todos.nivel_engajamento == "baixo").mean() * 100, "pct_possivel_churn": (todos.nivel_engajamento == "possível churn").mean() * 100,
    "pct_renovacao_desligada": todos.renovacao_desligada.astype(float).mean() * 100,
    "pct_influenciador": (todos.canal_aquisicao == "influenciador").mean() * 100, "tipo_principal_mais_comum": "-"}
print(tab.round(1).T)
print("\nnível de engajamento vazio entre os inativos em 90 dias:", int(ina.nivel_engajamento.isna().sum()))

assinantes ativos no corte: 14753 (esperado 14.753) | elegíveis: 13999 | inativos em 90 dias: 754
segmento                  Ativos ao ar livre Ativos em ambiente fechado Recém-chegados Esfriando Inativos em 90 dias (regra) Total de ativos
assinantes                              5001                       4030           2092      2876                         754           14753
pct_ativos                             33.90                      27.30          14.20     19.50                        5.10          100.00
treinos_por_semana                      2.60                       2.40           0.90      0.60                        0.00            1.80
dias_sem_treinar                        1.00                       1.00           2.00     18.00                      131.00            2.00
meses_de_casa                          10.80                      10.50           0.90     10.00                       15.50            8.60
pct_ar_livre                           80.80            

## 9. Gráfico de apoio: renovação desligada por segmento

Barras ordenadas, com a média dos ativos como referência.

In [10]:
r = tab.drop("Total de ativos").pct_renovacao_desligada.sort_values()
media = tab.loc["Total de ativos", "pct_renovacao_desligada"]
fig, ax = plt.subplots(figsize=(8, 3.6))
cores = ["#e5484d" if v > media else "#222" for v in r.values]
ax.barh([f"{i}  ({int(tab.loc[i, 'assinantes']):,} assinantes)".replace(",", ".") for i in r.index], r.values, color=cores)
ax.axvline(media, color="#e5484d", ls="--", lw=1); ax.text(media, len(r) - 0.4, f"média dos ativos {media:.1f}%".replace(".", ","), color="#e5484d", fontsize=8)
for y, v in enumerate(r.values): ax.text(v + 0.8, y, f"{v:.1f}%".replace(".", ","), va="center", fontsize=9)
ax.set_xlabel("% com renovação automática desligada em 30/09/2026"); ax.set_xlim(0, max(r.max() * 1.18, 10))
for s_ in ["top", "right"]: ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(S / "segmentacao_oficial_renovacao.png", dpi=130)
print("salvo: saidas/segmentacao_oficial_renovacao.png")

salvo: saidas/segmentacao_oficial_renovacao.png


## 10. Salvar

In [11]:
out = todos[["user_id", "segmento"]].merge(e[["user_id"]].assign(grupo_kmeans=pd.Series(lab).values), on="user_id", how="left")
out.to_parquet(P / "segmentos_oficiais.parquet", index=False)
tab.round(2).to_csv(S / "segmentacao_oficial_perfis.csv", encoding="utf-8-sig", index_label="segmento")
cent = pd.DataFrame(sc.inverse_transform(km4.cluster_centers_), columns=NUM)
cent.index = [NOMES[i] for i in range(K)]
cent.round(3).to_csv(S / "segmentacao_oficial_centros.csv", encoding="utf-8-sig", index_label="segmento")
print("segmentos:", out.shape, "->", out.segmento.value_counts().to_dict())
print("arquivos: dados/processados/segmentos_oficiais.parquet; saidas/segmentacao_oficial_perfis.csv, _centros.csv, _curva.csv, _curvas.png, _renovacao.png, _padronizacao.csv")

segmentos: (14753, 3) -> {'Ativos ao ar livre': 5001, 'Ativos em ambiente fechado': 4030, 'Esfriando': 2876, 'Recém-chegados': 2092, 'Inativos em 90 dias (regra)': 754}
arquivos: dados/processados/segmentos_oficiais.parquet; saidas/segmentacao_oficial_perfis.csv, _centros.csv, _curva.csv, _curvas.png, _renovacao.png, _padronizacao.csv
